# Prac 08.2

In this homework we will be working with the Fashion MNIST dataset. You will be given a classifier which suffers from considerable overfitting. Your objective will be to employ regularization techniques to mitigate the overfitting problem.

Let's start with the usual imports.

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Dense, Dropout, Flatten, Input, MaxPooling2D, BatchNormalization
from tensorflow.keras import Model
from time import time

from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]

# Set the seeds for reproducibility
from numpy.random import seed
from tensorflow.random import set_seed
seed_value = 1234578790
seed(seed_value)
set_seed(seed_value)

### Dataset

The MNIST fashgion dataset [link](https://github.com/zalandoresearch/fashion-mnist) was build by Zalando Reasearch tem consists of monochrome images of different type of clothing, namely:
* 0	T-shirt/top
* 1	Trouser
* 2	Pullover
* 3	Dress
* 4	Coat
* 5	Sandal
* 6	Shirt
* 7	Sneaker
* 8	Bag
* 9	Ankle boot

It is also one of the Keras built-in datasets. Let's load the images and quickly inspect it.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

# Dataset params
num_classes = 10
size = x_train.shape[1]

print('Train set:   ', len(y_train), 'samples')
print('Test set:    ', len(y_test), 'samples')
print('Sample dims: ', x_train.shape)

Let's visualise some random samples.

In [ ]:
cnt = 1
for r in range(3):
    for c in range(6):
        idx = np.random.randint(len(x_train))
        plt.subplot(3,6,cnt)
        plt.imshow(x_train[idx, ...], cmap='gray')
        plt.title(y_train[idx])
        cnt = cnt + 1

### Building the Classifier

We are now going to build the baseline classifier that you will use throughout this homework.

In [ ]:
from sklearn.model_selection import train_test_split

# Data normalization
x_train = x_train / 255
X_test = x_test / 255

X_train, X_val, y_train, y_val = train_test_split(
    x_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

In [ ]:
inputs = Input(shape=(28, 28, 1))
net = Conv2D(32, kernel_size=(3, 3), activation="relu", padding='same')(inputs)
net = Flatten()(net)
net = Dense(128)(net)
outputs = Dense(10, activation="softmax")(net)

model = Model(inputs, outputs)
model.summary()

In [ ]:
epochs = 50
batch_size = 64

model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
history = model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs, validation_data=(X_val, y_val))

In [ ]:
def plot_history(history):
    h = history.history
    epochs = range(len(h['loss']))

    plt.subplot(121), plt.plot(epochs, h['loss'], '.-', epochs, h['val_loss'], '.-')
    plt.grid(True), plt.xlabel('epochs'), plt.ylabel('loss')
    plt.legend(['Train', 'Validation'])
    plt.subplot(122), plt.plot(epochs, h['accuracy'], '.-',
                               epochs, h['val_accuracy'], '.-')
    plt.grid(True), plt.xlabel('epochs'), plt.ylabel('Accuracy')
    plt.legend(['Train', 'Validation'])

    print('Train Acc     ', h['accuracy'][-1])
    print('Validation Acc', h['val_accuracy'][-1])

plot_history(history)

As you can see, the classifier suffers from massive overfitting. The validation accuracy is around 88% while the training accuracy is close to 1.

### Combat the Overfitting!

Now it is your turn. Use the classifier as a baseline, include some regularization techniques and try to improve the classification performance. You can try any techniques you might see fit, e.g.,
* Dropout
* Batch normalization
* Weight regularization
* Data augmentation
* Early stopping
* Pooling
* Reducing the number of parameters (the size of the network)
* ...

There are to objective you shall fulfill in order to successfully complete this homework:
* The validation accuracy shall be above 91%
* Your network (with all the regularizations applied) shall **not** be larger than the baseline

In [ ]:
from tensorflow.keras import layers, models, callbacks, regularizers

inputs = layers.Input(shape=(28, 28, 1))

conv1       = layers.Conv2D(32, (3, 3), padding='same')(inputs)
batch1      = layers.BatchNormalization()(conv1)
elu1        = layers.Activation('elu')(batch1)
max_pool1   = layers.MaxPooling2D(pool_size=(2, 2))(elu1)
dropout1    = layers.Dropout(0.2)(max_pool1)

conv2       = layers.Conv2D(64, (3, 3), padding='same')(dropout1)
batch2      = layers.BatchNormalization()(conv2)
elu2        = layers.Activation('elu')(batch2)
max_pool2   = layers.MaxPooling2D(pool_size=(2, 2))(elu2)
dropout2    = layers.Dropout(0.3)(max_pool2)

conv3       = layers.Conv2D(128, (3, 3), padding='same')(dropout2)
batch3      = layers.BatchNormalization()(conv3)
elu3        = layers.Activation('elu')(batch3)
max_pool3   = layers.MaxPooling2D(pool_size=(2, 2))(elu3)
dropout3    = layers.Dropout(0.4)(max_pool3)


flat        = layers.Flatten()(dropout3)
dense1      = layers.Dense(128, activation='elu')(flat)
batch4      = layers.BatchNormalization()(dense1)
dropout4    = layers.Dropout(0.5)(batch4)
outputs     = layers.Dense(10, activation='softmax')(dropout4)

model = models.Model(inputs, outputs)

model.summary()

In [ ]:
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=5,
    min_lr=1e-6
)


In [ ]:
# Train the network
epochs = 50
batch_size = 64

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

history = model.fit(
    X_train, y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, reduce_lr] # Attach callbacks here
)

In [ ]:
# Show the results
plot_history(history)

In [ ]:
# Print the overall stats
ev = model.evaluate(X_test, y_test)
print('Test loss  ', ev[0])
print('Test metric', ev[1])

### Questions

* What have you done in order to improve the performance?
> Added more Conv layers to better capture hidden patterns, added Batch Normalization to stabilize input between layers, added Pooling layers to reduce size, added Dropout to avoid overfitting. And these imrpovements reduced the num of params from 3,213,002 to 242,954 and achieving valid acc from ~0.88-0.89 to ~0.93-0.94
* Have you tried configurations that did not work out?
> No, I added everything that I knew and what you recommended above))